1 — Data Prep\nGénère `dataset.parquet` depuis le feed MT5 du broker connecté.\n\n**Sortie** : `dataset.parquet` (1 ligne = 1 bougie H4 en état de tendance, 21 features, issue simulée nette de commission+spread).\n\n⚠️ Identifiants via variables d'environnement, jamais en dur.\n\n**Ensuite : envoie `dataset.parquet` pour validation avant tout entraînement.**

In [17]:
# pip install MetaTrader5 pandas numpy pyarrow
# Identifiants : fichier .env (MT5_LOGIN / MT5_PASSWORD / MT5_SERVER) dans ce dossier.
# ⚠️ Ne jamais committer/partager .env — il contient le mot de passe du compte.
import os

def load_dotenv_simple(path='.env'):
    """Charge .env dans os.environ (sans dependance externe)."""
    if not os.path.exists(path): return
    with open(path, encoding='utf-8-sig') as f:
        for line in f:
            line=line.strip()
            if not line or line.startswith('#') or '=' not in line: continue
            k,v=line.split('=',1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_dotenv_simple()
for v in ('MT5_LOGIN','MT5_PASSWORD','MT5_SERVER'):
    assert os.environ.get(v), f'variable manquante: {v} (renseigne .env ou $env:{v})'
print('identifiants OK | serveur:', os.environ['MT5_SERVER'])

identifiants OK | serveur: FTMO-Demo


In [18]:
# === CONFIG PARTAGEE ===
SYMBOLS  = ['EURJPY','USDJPY','GBPJPY','GBPUSD','EURUSD','NZDUSD','AUDJPY','AUDUSD']
SL_PIPS  = {'AUDJPY':38,'AUDUSD':28,'EURJPY':48,'EURUSD':32,
            'GBPJPY':59,'GBPUSD':42,'NZDUSD':28,'USDJPY':39}
SPREAD_PIPS = {'AUDJPY':0.7,'AUDUSD':0.4,'EURJPY':0.6,'EURUSD':0.2,
               'GBPJPY':0.8,'GBPUSD':0.4,'NZDUSD':0.5,'USDJPY':0.3}
RR=1.70; MAX_HOLD_H=168; START_YEAR=2001; SELECT_Q=0.95
# START_YEAR=2001 : profondeur maximale du feed FTMO (verifiee) —
#   H4/D1 disponibles depuis 2001 ; H1 seulement depuis ~mi-2010.
#   La resolution TP/SL utilise le H1 quand il existe, sinon repli H4
#   PRIORITE SL (conservateur) — voir cellule generation.
FEAT=['rsi14','pctB','bw','ext','adx14','atr_norm','body','up_wick','dn_wick','bull',
      'dist_hi20','dist_lo20','bars_since_flip','ret5_atr','d1_state','d1_rsi',
      'd1_aligned','hour','dow','dir_b','sym_c']
SYM_CODE={s:i for i,s in enumerate(sorted(SYMBOLS))}
DATASET='dataset.parquet'


In [20]:
# === INDICATEURS & FEATURES (source unique — NE PAS MODIFIER sans re-entrainer) ===
import numpy as np, pandas as pd

def _sma(s,n): return s.rolling(n).mean()

def _rsi(c,n=14):
    d=c.diff(); g=d.clip(lower=0); l=-d.clip(upper=0)
    ag=g.ewm(alpha=1/n,min_periods=n,adjust=False).mean()
    al=l.ewm(alpha=1/n,min_periods=n,adjust=False).mean()
    return 100-100/(1+ag/al.replace(0,np.nan))

def _atr(df,n=14):
    tr=pd.concat([df.high-df.low,(df.high-df.close.shift()).abs(),
                  (df.low-df.close.shift()).abs()],axis=1).max(axis=1)
    return tr.ewm(alpha=1/n,min_periods=n,adjust=False).mean()

def _adx(df,n=14):
    up=df.high.diff(); dn=-df.low.diff()
    plus=np.where((up>dn)&(up>0),up,0.0); minus=np.where((dn>up)&(dn>0),dn,0.0)
    trn=_atr(df,n)
    pdi=100*pd.Series(plus,index=df.index).ewm(alpha=1/n,adjust=False).mean()/trn
    mdi=100*pd.Series(minus,index=df.index).ewm(alpha=1/n,adjust=False).mean()/trn
    return (100*(pdi-mdi).abs()/(pdi+mdi)).ewm(alpha=1/n,adjust=False).mean()

def features_h4(df):
    df=df.copy()
    df['sma10']=_sma(df.close,10); df['sma50']=_sma(df.close,50)
    df['rsi14']=_rsi(df.close,14)
    m=df.close.rolling(20).mean(); sd=df.close.rolling(20).std(ddof=0)
    df['bb_up']=m+2.5*sd; df['bb_lo']=m-2.5*sd
    df['pctB']=(df.close-df.bb_lo)/(df.bb_up-df.bb_lo)
    df['bw']=(df.bb_up-df.bb_lo)/m
    df['atr14']=_atr(df,14); df['adx14']=_adx(df,14)
    df['ext']=(df.close-df.sma50).abs()/df.sma50
    df['atr_norm']=df.atr14/df.close
    rng=(df.high-df.low).replace(0,np.nan)
    df['body']=(df.close-df.open).abs()/rng
    df['up_wick']=(df.high-df[['open','close']].max(axis=1))/rng
    df['dn_wick']=(df[['open','close']].min(axis=1)-df.low)/rng
    df['bull']=(df.close>df.open).astype(int)
    df['dist_hi20']=(df.high.rolling(20).max()-df.close)/df.atr14
    df['dist_lo20']=(df.close-df.low.rolling(20).min())/df.atr14
    st=np.sign(df.sma10-df.sma50)
    # clip 300 : PARITE EA — l'EA ne voit que LOOKBACK=400 bougies, il ne peut
    # jamais produire un bars_since_flip > ~399 ; sans clip, le modele apprend
    # des splits sur des valeurs (milliers) que la prod ne genere jamais.
    # Le MEME clip est applique dans l'EA (MathMin(bsf,300)).
    df['bars_since_flip']=df.groupby((st!=st.shift()).cumsum()).cumcount().clip(upper=300)
    df['ret5_atr']=(df.close-df.close.shift(5))/df.atr14
    df['state']=st
    return df

def d1_context(d1):
    d1=d1.copy()
    d1['d1_state']=np.sign(_sma(d1.close,10)-_sma(d1.close,50))
    d1['d1_rsi']=_rsi(d1.close,14)
    return d1.set_index('time')[['d1_state','d1_rsi']]

In [21]:
# === CONNEXION MT5 ===
# ⚠️ IDENTIFIANTS VIA VARIABLES D'ENVIRONNEMENT UNIQUEMENT (jamais en dur).
# ⚠️ Construire le dataset sur le feed du broker de TRADING (FTMO) :
#    un feed different (ex. MetaQuotes-Demo) decoupe d'autres bougies H4
#    (fuseaux, gaps, week-ends) -> features differentes de la production.
import MetaTrader5 as mt5, os, time
from datetime import datetime, timedelta
assert mt5.initialize(login=int(os.environ['MT5_LOGIN']),
                      password=os.environ['MT5_PASSWORD'],
                      server=os.environ['MT5_SERVER']), mt5.last_error()
ai=mt5.account_info()
print('MT5 OK:',ai.login,'|',os.environ['MT5_SERVER'],'| equity',ai.equity,ai.currency)

def fetch(sym,tf,d0,d1,retry=3):
    # retry : sur une longue plage, le 1er appel declenche le telechargement
    # asynchrone de l'historique et peut renvoyer vide
    for _ in range(retry):
        r=mt5.copy_rates_range(sym,tf,d0,d1)
        if r is not None and len(r)>0:
            df=pd.DataFrame(r); df['time']=pd.to_datetime(df['time'],unit='s'); return df
        time.sleep(1.0)
    return None

def fetch_all(sym,tf,y0,d_end):
    # par TRANCHES ANNUELLES : copy_rates_range echoue silencieusement sur les
    # tres grandes plages, et renvoie vide si d0 precede la 1ere barre dispo
    parts=[]
    for y in range(y0,d_end.year+1):
        d=fetch(sym,tf,datetime(y,1,1),min(datetime(y+1,1,1),d_end))
        if d is not None: parts.append(d)
    if not parts: return None
    return (pd.concat(parts,ignore_index=True)
              .drop_duplicates('time').sort_values('time').reset_index(drop=True))


MT5 OK: 1513964555 | FTMO-Demo | equity 100002.26 USD


In [22]:
# === GENERATION DU DATASET (20-40 min : 25 ans x 8 paires) ===
def make_row(c,d1row,base):
    d1s=int(d1row.d1_state) if (d1row is not None and pd.notna(d1row.d1_state)) else 0
    return dict(rsi14=c.rsi14,pctB=c.pctB,bw=c.bw,ext=c.ext,adx14=c.adx14,
        atr_norm=c.atr_norm,body=c.body,up_wick=c.up_wick,dn_wick=c.dn_wick,
        bull=int(c.bull),dist_hi20=c.dist_hi20,dist_lo20=c.dist_lo20,
        bars_since_flip=int(c.bars_since_flip),ret5_atr=c.ret5_atr,d1_state=d1s,
        d1_rsi=(d1row.d1_rsi if (d1row is not None and pd.notna(d1row.d1_rsi)) else np.nan),
        d1_aligned=int(d1s!=0 and d1s==c.state),hour=c.time.hour,
        dow=int(c.time.dayofweek),dir_b=int(c.state>0),sym_c=SYM_CODE[base])

d1_end=datetime.utcnow();
rows=[]
for i,base in enumerate(SYMBOLS,1):
    if not mt5.symbol_select(base,True): print(f'[{i}] {base}: indisponible'); continue
    h4=fetch_all(base,mt5.TIMEFRAME_H4,START_YEAR,d1_end)
    h1=fetch_all(base,mt5.TIMEFRAME_H1,START_YEAR,d1_end)   # ne commence qu'en ~2010 (limite FTMO)
    dd=fetch_all(base,mt5.TIMEFRAME_D1,START_YEAR,d1_end)
    if h4 is None or dd is None: print(f'[{i}] {base}: pas de donnees H4/D1'); continue
    # h4res : OHLC H4 BRUT (avant dropna features) pour le repli de resolution
    h4res=h4.set_index('time')[['high','low','close']].sort_index()
    h1x=h1.set_index('time')[['high','low','close']].sort_index() if h1 is not None else None
    h1_min=h1x.index.min() if h1x is not None else pd.Timestamp.max
    h4=features_h4(h4); dix=d1_context(dd)
    pip=0.01 if 'JPY' in base else 0.0001; slp=SL_PIPS[base]*pip
    h4=h4.dropna(subset=['sma50','rsi14','adx14','atr14','bb_up']).reset_index(drop=True)
    n=0; n_fb=0
    for _,c in h4.iterrows():
        if c.state==0: continue
        long=c.state>0
        et=c.time+timedelta(hours=4); entry=c.close
        # --- RESOLUTION TP/SL : H1 quand disponible, sinon repli H4 ---
        # Repli H4 = PRIORITE SL (le SL est teste avant le TP dans chaque
        # bougie) : si les deux niveaux sont touches dans la meme bougie H4,
        # on compte la PERTE. Conservateur : sous-estime legerement le R des
        # periodes pre-2010, jamais l'inverse. (Teste : %TP 34 vs 32 sur
        # tranches comparables H4/H1.)
        if et>=h1_min: w=h1x.loc[et:et+timedelta(hours=MAX_HOLD_H)].reset_index()
        else:          w=h4res.loc[et:et+timedelta(hours=MAX_HOLD_H)].reset_index(); n_fb+=1
        sl=entry-slp if long else entry+slp
        tp=entry+RR*slp if long else entry-RR*slp
        R=np.nan
        for _,b in w.iterrows():
            if (long and b.low<=sl) or (not long and b.high>=sl): R=-1.0; break
            if (long and b.high>=tp) or (not long and b.low<=tp): R=RR; break
        else:
            if len(w):
                lc=w.iloc[-1].close
                R=round(((lc-entry)/slp) if long else ((entry-lc)/slp),3)
        # --- ANTI-LOOKAHEAD D1 (correctif critique) ---
        # Le D1 du jour J est indexe a J 00:00 mais sa CLOTURE n'est connue
        # qu'a J+1 00:00. Il n'est utilisable que si cette cloture est <= au
        # moment d'entree (et = c.time+4h), soit D1.time <= c.time-20h.
        # -> bougie H4 de 20:00 : D1 du jour meme (vient de clore) ;
        #    toutes les autres : D1 de la veille. Replique EXACTE de l'EA
        #    (CopyRates D1 shift=1 = jours clos uniquement).
        dr=dix.loc[:c.time-timedelta(hours=20)]; dr=dr.iloc[-1] if len(dr) else None
        rec=make_row(c,dr,base); rec.update(symbol=base,time=c.time,entry_time=et,R=R,
                                            res_tf=('H4' if et<h1_min else 'H1'))
        rows.append(rec); n+=1
        if n%2000==0: print(f'    {base}: {n}...')
    print(f'[{i}/{len(SYMBOLS)}] {base}: {n} lignes (dont {n_fb} resolues en H4 pre-{h1_min.year if h1x is not None else "?"})')
mt5.shutdown()


C:\Users\mbula\AppData\Local\Temp\ipykernel_15448\3413481336.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  d1_end=datetime.utcnow();


    EURJPY: 2000...
    EURJPY: 4000...
    EURJPY: 6000...
    EURJPY: 8000...
    EURJPY: 10000...
    EURJPY: 12000...
    EURJPY: 14000...
    EURJPY: 16000...
    EURJPY: 18000...
    EURJPY: 20000...
    EURJPY: 22000...
    EURJPY: 24000...
    EURJPY: 26000...
    EURJPY: 28000...
    EURJPY: 30000...
    EURJPY: 32000...
    EURJPY: 34000...
    EURJPY: 36000...
    EURJPY: 38000...
[1/8] EURJPY: 39701 lignes (dont 14650 resolues en H4 pre-2010)
    USDJPY: 2000...
    USDJPY: 4000...
    USDJPY: 6000...
    USDJPY: 8000...
    USDJPY: 10000...
    USDJPY: 12000...
    USDJPY: 14000...
    USDJPY: 16000...
    USDJPY: 18000...
    USDJPY: 20000...
    USDJPY: 22000...
    USDJPY: 24000...
    USDJPY: 26000...
    USDJPY: 28000...
    USDJPY: 30000...
    USDJPY: 32000...
    USDJPY: 34000...
    USDJPY: 36000...
    USDJPY: 38000...
[2/8] USDJPY: 39687 lignes (dont 14637 resolues en H4 pre-2010)
    GBPJPY: 2000...
    GBPJPY: 4000...
    GBPJPY: 6000...
    GBPJPY: 8000...
  

True

In [23]:
# === COUTS + SAUVEGARDE ===
df=pd.DataFrame(rows).dropna(subset=['R'])
df['comm_R']=df.symbol.map(lambda s:5.0/(SL_PIPS[s]*(7.0 if 'JPY' in s else 10.0)))
df['spread_R']=df.symbol.map(lambda s:SPREAD_PIPS[s]/SL_PIPS[s])
df['R_net']=df.R-df.comm_R-df.spread_R
df.to_parquet(DATASET,index=False)
print(len(df),'lignes ->',DATASET)
print('R_net moyen univers: {:+.4f} (attendu ~0, legerement negatif)'.format(df.R_net.mean()))
print('periode:',df.time.min(),'->',df.time.max())
print(df.symbol.value_counts().to_string())
# ==> ENVOIE dataset.parquet POUR VALIDATION AVANT L ETAPE 2 <==

317134 lignes -> dataset.parquet
R_net moyen univers: -0.0347 (attendu ~0, legerement negatif)
periode: 2001-01-11 04:00:00 -> 2026-07-13 08:00:00
symbol
EURJPY    39700
EURUSD    39698
GBPJPY    39691
AUDUSD    39687
USDJPY    39686
NZDUSD    39682
GBPUSD    39663
AUDJPY    39327
